# 10. Floating-Point Arithmetic, Conditioning & Numerical Stability

### Historical Lineage & Theoretical Foundations
- **Lineage:** *Welford (1962, Single-Pass Online Variance); Goldberg (1991, What Every Computer Scientist Should Know About Floating-Point Arithmetic); Neumaier (1974); Higham (2002, Accuracy and Stability of Numerical Algorithms)*.
- **Core Purpose:** Finite-precision floating-point arithmetic dynamics, matrix condition number diagnostics $\kappa(\mathbf{W})$, single-pass Welford online variance to prevent catastrophic cancellation, and numerically stable Log-Sum-Exp reductions.

### Decision Boundary: When NOT to Use
> [!WARNING]
> - **DO NOT evaluate sample variance via the naive formula $\mathbb{E}[X^2] - (\mathbb{E}[X])^2$:** When $\mu \gg \sigma$, subtracting two large, nearly identical quantities causes catastrophic cancellation, often producing negative variance or total loss of precision.
> - **DO NOT compute un-shifted $\log \sum \exp(z_i)$:** Logits exceeding $88.72$ in FP32 overflow to `+inf`, corrupting cross-entropy and attention softmax layers; always subtract $m = \max_i z_i$.
> - **DO NOT invert matrices without checking condition number $\kappa(\mathbf{A})$:** High condition numbers ($\kappa > 10^4$) amplify roundoff error by $\kappa(\mathbf{A})$, causing gradient oscillation or singular matrix crashes.


### Mathematical Formulation & Asymptotic Complexity

#### 1. Welford's Single-Pass Online Recurrence
$$M_k = M_{k-1} + \frac{x_k - M_{k-1}}{k}, \quad S_k = S_{k-1} + (x_k - M_{k-1})(x_k - M_k), \quad \sigma_N^2 = \frac{S_N}{N}$$

#### 2. Numerically Stable Log-Sum-Exp (LSE)
$$m = \max_{j \in \{1, \dots, D\}} z_j, \quad \operatorname{LSE}(\mathbf{z}) = m + \ln \left( \sum_{j=1}^D \exp(z_j - m) \right)$$

#### 3. Matrix Condition Number Formulation
$$\kappa_2(\mathbf{W}) = \frac{\sigma_{\max}(\mathbf{W})}{\sigma_{\min}(\mathbf{W})}$$

**Asymptotic Complexity:**
- Welford Recurrence: Time $\mathcal{O}(N \cdot D)$ single sequential pass; Space $\mathcal{O}(D)$ running accumulators.
- Stable Log-Sum-Exp: Time $\mathcal{O}(B \cdot D)$ bounded arithmetic; Space $\mathcal{O}(B)$.
- Condition Number: Time $\mathcal{O}(D_{\text{in}} D_{\text{out}} \min(D_{\text{in}}, D_{\text{out}}))$ SVD computation.


In [ ]:
from pathlib import Path

import torch
import torch.nn as nn

import core

env = core.setup_env(seed=42)
ckpt_path = Path("weights/00_foundations/10_floating_point_arithmetic_conditioning_and_numerical_stability.safetensors")
if not ckpt_path.is_file():
    ckpt_path = Path("../../weights/00_foundations/10_floating_point_arithmetic_conditioning_and_numerical_stability.safetensors")
weights, test_in, golden_out, meta = core.parity.load_reference_checkpoint(ckpt_path)
dim = int(meta.get("dim", 4))
batch_size = int(meta.get("batch_size", 8))
print(f"Device: {env.device} | Batch: {batch_size} | Dim: {dim} | Checkpoint: {ckpt_path.name}")


In [ ]:
def welford_mean_var(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    n, d = x.shape
    mean = torch.zeros(d, dtype=x.dtype, device=x.device)
    m2 = torch.zeros(d, dtype=x.dtype, device=x.device)
    for i in range(n):
        delta = x[i] - mean
        mean = mean + delta / float(i + 1)
        m2 = m2 + delta * (x[i] - mean)
    return mean, m2 / float(n)

def log_sum_exp_stable(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    m, _ = torch.max(x, dim=dim, keepdim=True)
    return m.squeeze(dim) + torch.log(torch.sum(torch.exp(x - m), dim=dim))

class StableLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_features, out_features))
        self.bias = nn.Parameter(torch.empty(out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight + self.bias

model = StableLinear(dim, 2)
with torch.no_grad():
    model.weight.copy_(weights["weight"])
    model.bias.copy_(weights["bias"])
print("StableLinear initialized with reference parameters.")


In [ ]:
# 1. Welford Online Statistics Parity vs Golden Reference
mean_pred, var_pred = welford_mean_var(test_in)
torch.testing.assert_close(mean_pred, weights["welford_mean"], atol=1e-6, rtol=1e-6)
torch.testing.assert_close(var_pred, weights["welford_var"], atol=1e-6, rtol=1e-6)

# 2. Linear Projection Forward Parity vs Golden Checkpoint
y_pred = model(test_in)
torch.testing.assert_close(y_pred, golden_out, atol=1e-5, rtol=1e-5)

# 3. Numerically Stable Log-Sum-Exp Parity
lse_pred = log_sum_exp_stable(y_pred, dim=-1)
torch.testing.assert_close(lse_pred, weights["lse_ref"], atol=1e-5, rtol=1e-5)

# 4. Autograd Gradient Parity
x_param = test_in.clone().detach().requires_grad_(True)
loss = 0.5 * torch.sum((model(x_param) - weights["target"]) ** 2)
loss.backward()
torch.testing.assert_close(model.weight.grad, weights["grad_weight"], atol=1e-5, rtol=1e-5)
torch.testing.assert_close(model.bias.grad, weights["grad_bias"], atol=1e-5, rtol=1e-5)
torch.testing.assert_close(x_param.grad, weights["grad_input"], atol=1e-5, rtol=1e-5)
print("All mathematical invariants (Welford + Linear + Stable LSE + Gradients) verified.")


In [ ]:
def run_welford():
    return welford_mean_var(test_in)

def run_torch_var():
    return torch.var(test_in, dim=0, unbiased=False)

def run_stable_lse():
    return log_sum_exp_stable(golden_out, dim=-1)

track_fns = {"Welford Single-Pass": run_welford, "PyTorch Native Var": run_torch_var, "Stable LogSumExp": run_stable_lse}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=batch_size)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# [!WARNING] PATHOLOGICAL MISUSE & FAILURE MODE: Catastrophic Cancellation in Variance
# 1. Flawed Anti-Pattern: E[X^2] - (E[X])^2 on shifted data suffers catastrophic cancellation
shift = 1e4
shifted_data = test_in + shift
naive_mean = torch.mean(shifted_data, dim=0)
naive_var = torch.mean(shifted_data ** 2, dim=0) - naive_mean ** 2

# True variance is shift-invariant: Var(X + c) == Var(X)
true_var = weights["welford_var"]
rel_err = float(torch.max(torch.abs(naive_var - true_var) / (true_var + 1e-8)).item())

# Active failure assertion proving catastrophic loss of precision
assert rel_err > 0.5, "Naive variance on shifted numbers must exhibit severe cancellation error"
print(f"[FAILURE DEMONSTRATED] Naive variance suffered {rel_err * 100:.1f}% relative error due to catastrophic cancellation!")

# 2. Correct Pattern: Welford recurrence maintains full precision on shifted data
w_mean_shift, w_var_shift = welford_mean_var(shifted_data)
torch.testing.assert_close(w_var_shift, true_var, atol=1e-3, rtol=1e-3)
print("[CORRECT] Welford recurrence preserved true variance with zero cancellation error.")

# 3. Compilation Verification
ok_exp, msg_exp = core.export.verify_torch_export(model, test_in)
ok_comp, msg_comp = core.export.verify_torch_compile_fullgraph(model, test_in)
assert ok_exp and ok_comp, "Compilation / export failed"
print("[CORRECT] Model verified with torch.compile(fullgraph=True) and exported.")


## Architectural Topology, Key Takeaways & Curriculum Navigation

### Numerical Stability Topology
<svg width="650" height="170" xmlns="http://www.w3.org/2000/svg" style="background:#0f172a; border-radius:8px; font-family:system-ui, sans-serif;">
  <defs>
    <marker id="arrow-s" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#38bdf8"/>
    </marker>
  </defs>
  <text x="30" y="25" fill="#94a3b8" font-size="12" font-weight="bold">FLOATING-POINT NUMERICS &amp; NUMERICAL STABILITY</text>
  <rect x="30" y="50" width="140" height="35" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="100" y="72" fill="#38bdf8" font-size="11" text-anchor="middle">Input Features [N, D]</text>
  <line x1="170" y1="67" x2="230" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-s)"/>
  <rect x="230" y="50" width="180" height="35" rx="6" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="320" y="72" fill="#10b981" font-size="11" text-anchor="middle">Welford Online Recurrence</text>
  <line x1="410" y1="67" x2="470" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-s)"/>
  <rect x="470" y="50" width="150" height="35" rx="6" fill="#1e293b" stroke="#a855f7" stroke-width="1.5"/>
  <text x="545" y="72" fill="#a855f7" font-size="11" text-anchor="middle">Stable Log-Sum-Exp</text>
  <rect x="30" y="110" width="590" height="35" rx="6" fill="#0f172a" stroke="#f59e0b" stroke-width="1.5" stroke-dasharray="4 4"/>
  <text x="325" y="132" fill="#f59e0b" font-size="11" text-anchor="middle">Stable Numerics: Zero Catastrophic Cancellation + Finite Bounded Exponents</text>
</svg>

### Key Takeaways
1. **Catastrophic Cancellation Immunity:** Always use Welford's recurrence for running statistics on data with non-zero or large mean offsets.
2. **Log-Sum-Exp Protection:** Max-subtraction prevents FP32 exponent overflow, ensuring finite log-probabilities and gradients.
3. **Conditioning Diagnostics:** Monitor matrix condition numbers $\kappa(\mathbf{W})$ to detect stiff loss surfaces before training collapse.

---
**Curriculum Navigation:**  
&larr; [Previous: 09. Nested, Ragged & Specialized Sparse Tensors](09_nested_ragged_and_specialized_sparse_tensors.ipynb) | [Master Index: README.md](../../README.md) | [Next: 11. Randomized Numerical Linear Algebra, Spectral Autograd & Hutch++](11_randomized_numerical_linear_algebra_spectral_autograd_and_hutchpp.ipynb) &rarr;
